# Mellyn Security — Colab A100 Open-Weight Evaluation Suite

Runs all three external security harnesses locally via **vLLM** (`http://127.0.0.1:8000/v1`) with automatic GPU VRAM model selection (**A100 80GB** `Qwen/Qwen3-30B-A3B-Instruct-2507`, **A100 40GB** `Qwen/Qwen3-32B-AWQ`, **L4 24GB** `Qwen/Qwen3-14B-AWQ`, **T4 16GB** `Qwen/Qwen3-8B-AWQ`):
1. **`mcp-fuzzer` (`evals/fuzzer/`)** — Protocol fuzzing & adversarial canary/Unicode/path probes against `mellyn.airlock.proxy`.
2. **`Promptfoo` (`evals/promptfoo/`)** — Side-by-side A/B comparison (**Mellyn OFF `direct` vs. Mellyn ON `airlock`**) across 12 prompt-injection, canary-exfiltration, SSRF, guardrail-tampering, and utility cases.
3. **`AgentDojo` (`evals/agentdojo/`)** — Quantitative Benign Utility vs. Attack Success Rate (ASR) across `workspace`, `banking`, `slack`, and `travel` suites (**Baseline vs. `MellynToolGate`**).
4. **`mellyn-ir`** — Cryptographic hash-chain and incident containment verification over every generated `audit.jsonl` ledger.


In [ ]:
%%bash
set -euo pipefail
nvidia-smi

# Commit to evaluate: the full 40-character SHA (git cannot fetch an abbreviated one). Pinning a commit keeps
# runs comparable; set a branch name instead to test unmerged changes. 1dbf831 = main after PR #42.
TARGET_REF="1dbf8311b51d281dccc551209e8b795ef54dab8d"

if [ ! -d "mellyn_security" ] && [ ! -f "pyproject.toml" ]; then
  git clone https://github.com/OramramLLC/mellyn_security.git
  git -C mellyn_security fetch --quiet origin "$TARGET_REF"
  git -C mellyn_security checkout --quiet --detach FETCH_HEAD
fi
if [ -d "mellyn_security" ]; then
  cd mellyn_security
fi
if [ ! -d "evals" ]; then
  echo "evals/ is missing from this checkout; set TARGET_REF to a commit that contains it." >&2
  exit 1
fi
mkdir -p outputs
git rev-parse HEAD > outputs/target_commit.txt 2>/dev/null || echo "not-a-git-checkout" > outputs/target_commit.txt
echo "Evaluating commit: $(cat outputs/target_commit.txt)"

python -m pip install -q --upgrade pip
# agentdojo is pinned because evals/agentdojo is written against the 0.1.35 API.
python -m pip install -q -e . vllm mcp-fuzzer agentdojo==0.1.35
node -v && npm -v


In [ ]:
import http.client
import os
import shutil
import subprocess  # noqa: S404 - queries nvidia-smi and launches local vLLM
import sys
import time
from pathlib import Path

if Path("mellyn_security").is_dir():
    os.chdir("mellyn_security")


def _select_vllm_profile() -> tuple[str, list[str]]:
    vram_mb = 40960
    cap = 8.0
    smi_bin = shutil.which("nvidia-smi")
    if smi_bin:
        q = subprocess.run(  # noqa: S603 - fixed read-only GPU query flags
            [smi_bin, "--query-gpu=memory.total,compute_cap", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=False,
        )
        first = next((ln for ln in q.stdout.splitlines() if ln.strip()), "")
        parts = [p.strip() for p in first.split(",")]
        if len(parts) >= 2:
            vram_mb, cap = int(float(parts[0])), float(parts[1])
    override = os.environ.get("MELLYN_EVAL_MODEL", "").strip()
    if vram_mb >= 75000:
        default_model, max_len, seqs, extra = "Qwen/Qwen3-30B-A3B-Instruct-2507", "16384", None, []
    elif vram_mb >= 36000:
        default_model = "Qwen/Qwen3-32B-AWQ"
        max_len, seqs = "16384", None
        extra = ["--reasoning-parser", "qwen3", "--default-chat-template-kwargs", '{"enable_thinking": false}']
    elif vram_mb >= 20000:
        default_model = "Qwen/Qwen3-14B-AWQ"
        max_len, seqs = "16384", "4"
        extra = ["--reasoning-parser", "qwen3", "--default-chat-template-kwargs", '{"enable_thinking": false}']
    else:
        default_model = "Qwen/Qwen3-8B-AWQ"
        max_len, seqs = "8192", "4"
        extra = ["--reasoning-parser", "qwen3", "--default-chat-template-kwargs", '{"enable_thinking": false}']
        if cap < 8.0:
            extra.extend(["--dtype", "half"])
    model = override or default_model
    flags = [
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--max-model-len",
        max_len,
        "--gpu-memory-utilization",
        "0.90",
    ]
    if seqs:
        flags.extend(["--max-num-seqs", seqs])
    flags.extend(extra)
    print(f"Detected {vram_mb} MiB VRAM (SM {cap}); using {model}")
    return model, flags


MODEL_NAME, VLLM_FLAGS = _select_vllm_profile()
BASE_URL = "http://127.0.0.1:8000/v1"
os.environ["OPENAI_BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["MELLYN_EVAL_MODEL"] = MODEL_NAME

Path("outputs").mkdir(parents=True, exist_ok=True)
vllm_log = open("outputs/vllm_server.log", "w", encoding="utf-8")

vllm_cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    MODEL_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    "8000",
    *VLLM_FLAGS,
]

vllm_proc = subprocess.Popen(  # noqa: S603 - starts local pinned vLLM module with static model flags
    vllm_cmd,
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
)

for attempt in range(90):
    if vllm_proc.poll() is not None:
        raise RuntimeError(f"vLLM exited early with code {vllm_proc.returncode}. Inspect outputs/vllm_server.log.")
    conn = http.client.HTTPConnection("127.0.0.1", 8000, timeout=2.0)
    try:
        conn.request("GET", "/v1/models")
        resp = conn.getresponse()
        if resp.status == 200:
            print(f"vLLM ready with {MODEL_NAME} after {attempt * 5}s")
            break
    except (OSError, http.client.HTTPException):
        time.sleep(5)
    finally:
        conn.close()
else:
    raise RuntimeError("vLLM failed to become healthy. Inspect outputs/vllm_server.log.")

In [ ]:
%%bash
set -euo pipefail
[ -d "mellyn_security" ] && cd mellyn_security
python -m evals.fuzzer.run_fuzzer --mode smoke --output-dir outputs/fuzzer_smoke
python -m evals.fuzzer.run_fuzzer --mode mcp-fuzzer --runs 25 --output-dir outputs/fuzzer_live


In [ ]:
%%bash
set -euo pipefail
[ -d "mellyn_security" ] && cd mellyn_security
mkdir -p outputs/promptfoo
# Promptfoo exits non-zero when baseline Direct MCP (Mellyn OFF) fails security tests; verify results.json exists.
PROMPTFOO_PYTHON="$(which python)" npx -y promptfoo@latest eval \
  -c evals/promptfoo/promptfooconfig.yaml -o outputs/promptfoo/results.json --no-cache -j 1 || true
test -s outputs/promptfoo/results.json


In [ ]:
%%bash
set -euo pipefail
[ -d "mellyn_security" ] && cd mellyn_security
python -m evals.agentdojo.run_agentdojo --suites workspace banking slack travel --output-dir outputs/agentdojo
cat outputs/agentdojo/agentdojo_scorecard.md


In [ ]:
import subprocess
import sys
from pathlib import Path

ledgers = sorted(Path("outputs").rglob("*audit.jsonl"))
print(f"Verifying {len(ledgers)} audit ledger(s)...")
for ledger in ledgers:
    res = subprocess.run(  # noqa: S603 - invokes pinned local mellyn.ir_lab module over output ledger paths
        [sys.executable, "-m", "mellyn.ir_lab.reconstruct", "--json", str(ledger)],
        capture_output=True,
        text=True,
        check=False,
    )
    print(f"{ledger}: exit={res.returncode}")
    if res.returncode != 0:
        print(res.stdout or res.stderr)